# Step 2 Helper: Applicant Name Harmonization Benchmark & UAV Competitive Landscape

**Team 28 · Flying Drones · Client: Leonardo S.p.A.**

This notebook directly resolves the Step 2 applicant-name harmonization bottleneck (Tutorial 11, Section 2):
1. **Part A (Runs Offline — No TIP Needed):** Benchmarks our zero-leakage multi-stage harmonizer (`src/harmonize.py`) against Raw PSN and Tutorial 11 Method A (Dictionary) on Prof. Ascione's official `PATSTAT_500_label_datav4.csv` gold standard (3,192 PATSTAT names across 481 organizations).
2. **Part B (Runs on TIP):** Applies the validated harmonizer + UAV corporate M&A tree (`config/sector.yaml`) to Team 28's locked Step 0 UAV families, outputs a manual audit table of the Top 50 merged clusters, and computes all six Liston-Heyes & Pilkington (2004) inventive concentration measures (`CR4`, `CR8`, `HH`, `1/HH`, `CV`, `Gini`, `Lotka L`).


## Part A — Offline Gold-Standard Benchmark (`PATSTAT_500_label_datav4.csv`)


In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from src.config_loader import load_config
from src.harmonize import (
    clean_with_tutorial_dictionary,
    evaluate_harmonization,
    harmonize_applicant_names,
    six_concentration_measures,
)

cfg = load_config(REPO_ROOT / "config" / "sector.yaml")

# Locate PATSTAT_500_label_datav4.csv (adjust path if located in innovation-management-datascience26/tutorial_1)
GOLD_CANDIDATES = [
    REPO_ROOT / "data" / "PATSTAT_500_label_datav4.csv",
    REPO_ROOT.parent / "innovation-management-datascience26" / "tutorial_1" / "PATSTAT_500_label_datav4.csv",
    Path("PATSTAT_500_label_datav4.csv"),
]
gold_path = next((p for p in GOLD_CANDIDATES if p.exists()), None)

if gold_path is not None:
    gold = pd.read_csv(gold_path).rename(columns={"comp": "name", "Group": "gold_group"})
    print(f"Loaded gold standard: {gold_path} ({len(gold):,} names, {gold['gold_group'].nunique():,} orgs)")

    r_raw = evaluate_harmonization(gold["gold_group"], gold["name"])
    r_dict = evaluate_harmonization(gold["gold_group"], gold["name"].map(clean_with_tutorial_dictionary))
    # Zero-leakage benchmark: apply_domain_aliases=False (no hardcoded company rules)
    m_multi = harmonize_applicant_names(gold["name"].tolist(), apply_domain_aliases=False)
    r_multi = evaluate_harmonization(gold["gold_group"], gold["name"].map(m_multi))

    bench = pd.DataFrame([
        {"Method": "Raw PSN (baseline)", **r_raw},
        {"Method": "Method A (Tutorial Dictionary)", **r_dict},
        {"Method": "Team 28 Multi-Stage Harmonizer", **r_multi},
    ]).set_index("Method")
    display(bench.round(4))
else:
    print("PATSTAT_500_label_datav4.csv not found locally. Place it in data/ or pull innovation-management-datascience26.")


Loaded gold standard: /home/jovyan/Patent-study/data/PATSTAT_500_label_datav4.csv (3,192 names, 481 orgs)


/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)


,groups_found,gold_groups,precision,recall,F1,homogeneity,completeness,V_measure
Method,,,,,,,,
Raw PSN (baseline),3192,481,1.0000,0.0000,0.0000,1.0000,0.5935,0.7449
Method A (Tutorial Dictionary),3010,481,0.9952,0.0022,0.0045,0.9999,0.5996,0.7496
Team 28 Multi-Stage Harmonizer,616,481,0.9747,0.9227,0.9480,0.9864,0.9531,0.9694


In [2]:
%pip install -q cleanco
from cleanco import basename


def clean_with_cleanco(name):
    "Method B (Tutorial 11): cleanco removes legal forms, then the Method A cleaning."
    return clean_with_tutorial_dictionary(basename(str(name)))


def first_word(name):
    "Omar's method: keep only the first word of the name."
    words = str(name).lower().split()
    return words[0] if words else ""


r_cleanco = evaluate_harmonization(gold["gold_group"], gold["name"].map(clean_with_cleanco))
r_first = evaluate_harmonization(gold["gold_group"], gold["name"].map(first_word))

bench_full = pd.DataFrame([
    {"Method": "Raw PSN (baseline)", **r_raw},
    {"Method": "Method A (Tutorial Dictionary)", **r_dict},
    {"Method": "Method B (cleanco)", **r_cleanco},
    {"Method": "First word only", **r_first},
    {"Method": "Team 28 Multi-Stage Harmonizer", **r_multi},
]).set_index("Method")
display(bench_full.round(4))

# save the table for the Technical Appendix (A3)
OUT_STEP2 = REPO_ROOT / "outputs" / "step2_competitive_landscape"
OUT_STEP2.mkdir(parents=True, exist_ok=True)
bench_full.round(4).to_csv(OUT_STEP2 / "harmonization_benchmark.csv")
print(f"Saved to {OUT_STEP2 / 'harmonization_benchmark.csv'}")


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/opt/conda/lib/python3.12/site-packages/sklearn/metrics/cluster/_supervised.py:50: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(labels_pred)


,groups_found,gold_groups,precision,recall,F1,homogeneity,completeness,V_measure
Method,,,,,,,,
Raw PSN (baseline),3192,481,1.0000,0.0000,0.0000,1.0000,0.5935,0.7449
Method A (Tutorial Dictionary),3010,481,0.9952,0.0022,0.0045,0.9999,0.5996,0.7496
Method B (cleanco),2984,481,0.9960,0.0027,0.0054,0.9999,0.6006,0.7504
First word only,627,481,0.9658,0.8758,0.9186,0.9787,0.9345,0.9561
Team 28 Multi-Stage Harmonizer,616,481,0.9747,0.9227,0.9480,0.9864,0.9531,0.9694


Saved to /home/jovyan/Patent-study/outputs/step2_competitive_landscape/harmonization_benchmark.csv


## Part B — Run on EPO TIP: Harmonize Team 28 UAV Applicants & Compute Concentration

Set `RUN_TIP_EXTRACTION = True` when executing inside EPO TIP. It automatically loads the same locked Step 0 family CSV (`uav_final_query_families.csv`), harmonizes all UAV applicants, prints the **Top 50 merged clusters for manual inspection** (Tutorial 11 §2.2), and computes the six concentration measures.


In [3]:
RUN_TIP_EXTRACTION = True   # Flip to True on EPO TIP
ENV = "TEST"
STEP0_CSV_PATH = None

if RUN_TIP_EXTRACTION:
    from epo.tipdata.patstat import PatstatClient
    from epo.tipdata.patstat.database.models import TLS201_APPLN, TLS206_PERSON, TLS207_PERS_APPLN
    from sqlalchemy import func
    from src.config_loader import get_last_complete_year
    from src.step0_loader import get_step0_families, iter_batches

    LAST_COMPLETE_YEAR = get_last_complete_year(cfg)
    YEAR_MIN = int(cfg["time_window"]["year_min"])
    IPR_TYPE = cfg["counting_rules"]["ipr_type"]
    OUT_STEP2 = REPO_ROOT / "outputs" / "step2_competitive_landscape"
    OUT_STEP2.mkdir(parents=True, exist_ok=True)

    patstat = PatstatClient(env=ENV)
    db = patstat.orm()

    step0_df, src_desc = get_step0_families(patstat, db, cfg, override_csv_path=STEP0_CSV_PATH)
    fids = step0_df["docdb_family_id"].astype(int).tolist()
    print(f"Step 2 loaded {len(fids):,} UAV families from {src_desc}")

    frames = []
    for batch in iter_batches(fids, batch_size=5000):
        q = (
            db.query(
                TLS201_APPLN.docdb_family_id.label("unit_id"),
                func.min(TLS201_APPLN.earliest_filing_year).label("year"),
                TLS206_PERSON.psn_id,
                TLS206_PERSON.psn_name,
                TLS206_PERSON.psn_sector,
                func.trim(TLS206_PERSON.person_ctry_code).label("ctry"),
            )
            .join(TLS207_PERS_APPLN, TLS201_APPLN.appln_id == TLS207_PERS_APPLN.appln_id)
            .join(TLS206_PERSON, TLS207_PERS_APPLN.person_id == TLS206_PERSON.person_id)
            .filter(
                TLS201_APPLN.ipr_type == IPR_TYPE,
                TLS207_PERS_APPLN.applt_seq_nr > 0,
                TLS201_APPLN.docdb_family_id.in_(batch),
            )
            .group_by(
                TLS201_APPLN.docdb_family_id,
                TLS206_PERSON.psn_id,
                TLS206_PERSON.psn_name,
                TLS206_PERSON.psn_sector,
                TLS206_PERSON.person_ctry_code,
            )
        )
        chunk = patstat.df(q)
        if chunk is not None and not chunk.empty:
            frames.append(chunk)

    raw_applts = pd.concat(frames, ignore_index=True)
    raw_comp = raw_applts[raw_applts["year"].between(YEAR_MIN, LAST_COMPLETE_YEAR)].copy()

       # Apply multi-stage harmonizer + UAV corporate tree consolidation
    hmap = harmonize_applicant_names(
        raw_comp["psn_name"].dropna().unique(),
        domain_groups=cfg["domain_corporate_groups"],
        apply_domain_aliases=True,
    )
    hmap.update({n: "ALPHABET" for n in hmap if "X开发" in str(n)})   # Chinese name of X Development
    raw_comp["applicant"] = raw_comp["psn_name"].map(hmap)

    # Individuals are people, not organisations: never merge them by surname (e.g. LEE, WANG)
    is_person = raw_comp["psn_sector"].astype(str).str.upper().str.contains("INDIVIDUAL")
    raw_comp.loc[is_person, "applicant"] = raw_comp.loc[is_person, "psn_name"]

    # Top 50 manual audit table: inspect merged PSN variants before freezing Step 2!
    audit_top50 = (
        raw_comp.groupby("applicant")
        .agg(
            families=("unit_id", "nunique"),
            n_psn_variants=("psn_name", "nunique"),
            sample_variants=("psn_name", lambda s: " | ".join(sorted(set(s))[:5])),
            top_country=("ctry", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else ""),
            top_sector=("psn_sector", lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else ""),
        )
        .sort_values("families", ascending=False)
        .head(50)
    )
    audit_top50.to_csv(OUT_STEP2 / "top50_harmonized_applicants_audit.csv")
    display(audit_top50.head(25))

    # Compare concentration BEFORE vs AFTER harmonization (great Technical Appendix table!)
    fam_raw = raw_comp.groupby("psn_name")["unit_id"].nunique().to_numpy()
    fam_harm = raw_comp.groupby("applicant")["unit_id"].nunique().to_numpy()
    conc_compare = pd.DataFrame({
        "Raw PSN": six_concentration_measures(fam_raw),
        "Harmonized (Team 28)": six_concentration_measures(fam_harm),
    })
    conc_compare.to_csv(OUT_STEP2 / "concentration_raw_vs_harmonized.csv")
    display(conc_compare.round(4))


Step 2 loaded 1,624 UAV families from live_yaml_query


,families,n_psn_variants,sample_variants,top_country,top_sector
applicant,,,,,
ALPHABET,69,6,GOOGLE | MAKANI POWER | MAKANI TECHNOLOGIES | ...,US,COMPANY
BOEING,49,7,AURORA FLIGHT SCIENCES CORPORATION | AURORA FL...,US,COMPANY
AMAZON TECHNOLOGIES,41,1,AMAZON TECHNOLOGIES,US,COMPANY
WALMART APOLLO,36,1,WALMART APOLLO,US,COMPANY
DJI,30,2,SZ DJI TECHNOLOGY COMPANY | エスゼット ディージェイアイ オスモ...,CN,COMPANY
STATE GRID CORP OF CHINA,26,27,"BEIXIANG COUNTY POWER SUPPLY COMPANY, STATE GR...",,COMPANY
AIRBUS,17,8,AIRBUS (S.A.S.) | AIRBUS DEFENCE AND SPACE | A...,DE,COMPANY
TEXTRON,16,4,BELL HELICOPTER TEXTRON | BELL TEXTRON | TEXTR...,US,COMPANY
LOCKHEED MARTIN,15,3,LOCKHEED MARTIN CORPORATION | SIKORSKY AIRCRAF...,US,COMPANY


,Raw PSN,Harmonized (Team 28)
n_applicants,1432.0000,1274.0000
CR4_pct,7.3903,9.7112
CR8_pct,12.0554,14.1434
HH,0.0029,0.0041
numbers_equivalent_1_over_HH,341.9585,242.6907
CV,1.7860,2.0622
Gini,0.3169,0.3424
Lotka_L,-1.6129,-1.5458
Lotka_p_value,0.0000,0.0000


In [4]:
# Where does the client stand? Families, rank and the PATSTAT names merged into it
fam_by_applicant = raw_comp.groupby("applicant")["unit_id"].nunique().sort_values(ascending=False)
ranks = fam_by_applicant.rank(ascending=False, method="min").astype(int)

CLIENT = "LEONARDO"
if CLIENT in fam_by_applicant.index:
    print(f"{CLIENT}: {fam_by_applicant[CLIENT]} families, rank {ranks[CLIENT]} of {len(fam_by_applicant):,} applicants")
    display(raw_comp.loc[raw_comp["applicant"] == CLIENT, "psn_name"].value_counts().to_frame("rows"))
else:
    print(f"{CLIENT} not found in this dataset (expected on TEST, which is only a sample)")

LEONARDO: 5 families, rank 35 of 1,274 applicants


,rows
psn_name,
OTO MELARA,2
LEONARDO,2
SELEX COMMUNICATIONS,1
LEONARDO MW,1
